# CrisisGraph ML-S1, session 2: guidance accuracy and the llama.cpp reference

Plan ML-S1 (approved 2026-11-02). Session 1 (2026-11-03/04, rerun 2026-11-05 as session 1b) measured
variant **B**: vLLM 0.11.0, FP16 base + SFT-V4 LoRA at run time, one T4, Triton attention.

- B accuracy (repeated exactly in two runs): 120 usable, critical under-triage 6, V2 exact 85,
  review recall 0.925, **invented-fact cases 35** (gate <= 34, local 31): B misses one gate.
- Throughput is limited by the grammar library vLLM chose (`auto`, xgrammar): about 12 extractions/min
  at concurrency 32. With **guidance** it was 39.5/min, and its answers matched B on the 64 cases benchmarked.
- Merged weights (variant C) and AWQ (variant D) are dropped: the runtime adapter costs only about 20%.

This session answers the two remaining questions:

| Step | Variant | Question | Time |
|---|---|---|---|
| 2 | **B-guidance** | Does B with guidance pass the gates on all 120 cases? | ~35 min |
| 3 | **A** | llama.cpp CUDA with the locally validated files: are B's extra invented facts from the FP16 weights or the engine? | ~45 min |

**Run it unattended:** settings → Accelerator **GPU T4** (one GPU is used), Internet **on**, attach the
private dataset `crisisgraph-ml-s1` (version from commit 950eb3f), then Save Version → **Save & Run All**.
`RUN` in step 0 picks the steps. Step 3 records an error and continues instead of stopping the run.
Afterwards download `ml-s1-session2-results.zip` from the version's Output tab.

**Privacy:** keep this notebook and the dataset private. The blind set is only sent to the model
served inside this notebook, never to a hosted API.

**Parity gates** (declared in the plan): 120/120 usable outputs, critical under-triage <= 8,
V2 exact >= 78, invented-fact cases <= 34, review recall >= 0.89.

## 0. Settings

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

# Pinned versions. VLLM_VERSION is checked in step 1; if it does not run on the T4,
# change it here, rerun from this cell, and note the change in the report.
VLLM_VERSION = "0.11.0"
# vLLM 0.11 does not cap transformers, and transformers 5 removed a tokenizer attribute it uses
# (all_special_tokens_extended). Pin a 4.x release inside vLLM's range (>= 4.56).
TRANSFORMERS_VERSION = "4.57.1"
BASE_REPO = "Qwen/Qwen3-4B-Instruct-2507"
BASE_REV = "cdbee75f17c01a7cc42f958dc650907174af0554"      # same revision as SFT-V4 training
GGUF_REPO = "lmstudio-community/Qwen3-4B-Instruct-2507-GGUF"
GGUF_REV = "4edb920b6f14e3b9284d4502a6485103d72cde05"
GGUF_FILE = "Qwen3-4B-Instruct-2507-Q4_K_M.gguf"
GGUF_SHA256 = "8cdb57cbb880d313736a9bc4e3d3d2485f145b5e19cf33783746e753e82641fc"
LLAMA_CPP_TAG = "b11429"                                    # build used for the local SFT-V4 evaluation
LLAMA_CPP_COMMIT_PREFIX = "d812350493"
# Variant D: a published AWQ of the same base. Chosen in step 5 from the listed candidates.
AWQ_REPO = None
AWQ_REV = None

GATES = {"usable_outputs": (">=", 120), "critical_under_triage": ("<=", 8), "v2_exact_match": (">=", 78),
         "invented_fact_cases": ("<=", 34), "v2_review_recall": (">=", 0.89)}

WORK = Path("/kaggle/working")
RUNS = WORK / "runs"                 # results: kept and downloaded
SCRATCH = Path("/tmp/ml-s1")         # models, venv, builds: large, not kept (20 GB cap on /kaggle/working)
for p in (RUNS, SCRATCH):
    p.mkdir(parents=True, exist_ok=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4
# On the T4 (no FlashAttention), vLLM 0.11 picks FlexAttention: measured 9.1 tokens/s for the plain base
# model against 19.4 with Triton attention (session 1, 2026-11-03). FlashInfer is not installed with vLLM 0.11.
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

# Which steps run, so the whole notebook can be run with Run All or as a saved background version
# (Save Version > Save & Run All). Session 2: B with the guidance grammar library, then variant A.
RUN = {"B_guidance": True, "A": True}

## 1a. Bundle: verify and install the evaluator

In [ ]:
found = glob.glob("/kaggle/input/**/BUNDLE.json", recursive=True)
assert found, "attach the private dataset crisisgraph-ml-s1 (Add input)"
SRC = Path(found[0]).parent
BUNDLE = SCRATCH / "bundle"
if not BUNDLE.exists():
    shutil.copytree(SRC, BUNDLE)
bad = subprocess.run(["sha256sum", "-c", "--quiet", "SHA256SUMS"], cwd=BUNDLE, capture_output=True, text=True)
assert bad.returncode == 0, bad.stdout + bad.stderr
print("bundle checksums ok:", json.loads((BUNDLE / "BUNDLE.json").read_text())["git_commit"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(BUNDLE / "ml")], check=True)

ML = BUNDLE / "ml"
DATASET = ML / "data" / "triage-eval-v3.1.0.jsonl"
PROMPT = ML / "prompts" / "triage-extraction-v3.2.txt"
SCHEMA = BUNDLE / "schemas" / "triage-extraction-v3.json"
LEAK_AUDIT = ML / "data" / "triage-sft-v4-leak-audit.json"
ADAPTER_PEFT = BUNDLE / "adapter" / "peft"
ADAPTER_GGUF = BUNDLE / "adapter" / "adapter-f16.gguf"
# The running kernel does not see an editable install made after it started (new processes do).
if str(ML / "src") not in sys.path:
    sys.path.insert(0, str(ML / "src"))
import crisisgraph_ml
print("crisisgraph_ml from", crisisgraph_ml.__file__)
from crisisgraph_ml import triage_eval
assert hasattr(triage_eval, "grammar_schema"), (
    "this dataset version was built before commit 950eb3f (vLLM rejects uniqueItems): "
    "upload ml/internal/kaggle/crisisgraph-ml-s1.zip as a new dataset version and attach it")

## 1b. Environment record

In [ ]:
def sh(cmd, **kw):
    """Run a shell command, return stdout (raises on failure)."""
    return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True, **kw).stdout.strip()

env = {
    "recorded": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv,noheader"),
    "cuda_runtime_nvcc": sh("nvcc --version | tail -1 || true"),
    "python": sys.version,
    "cpu": sh("lscpu | grep 'Model name' || true"),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1),
    "attention_backend": os.environ.get("VLLM_ATTENTION_BACKEND"),
}
print(json.dumps(env, indent=2))

## 1c. Install the pinned vLLM in its own environment
Kaggle's own torch stays untouched; vLLM runs from `/tmp/ml-s1/venv-vllm`.

In [ ]:
VENV = SCRATCH / "venv-vllm"
VPY = VENV / "bin" / "python"
if not VPY.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    sh(f"uv venv --python 3.12 {VENV}")
# Always run (fast when already satisfied), so a changed pin also fixes an existing environment.
sh(f'uv pip install --python {VPY} "vllm=={VLLM_VERSION}" "transformers=={TRANSFORMERS_VERSION}" peft huggingface_hub')
env["vllm_stack"] = {k: sh(f"{VPY} -c 'import {k}; print({k}.__version__)'") for k in ("vllm", "torch", "transformers", "peft")}
env["torch_cuda"] = sh(f"{VPY} -c 'import torch; print(torch.version.cuda, torch.cuda.get_device_capability())'")
print(env["vllm_stack"], env["torch_cuda"])

## 1d. Base model at the pinned revision

In [ ]:
from huggingface_hub import snapshot_download
BASE_DIR = Path(snapshot_download(BASE_REPO, revision=BASE_REV, local_dir=SCRATCH / "base"))
print(BASE_DIR, sh(f"du -sh {BASE_DIR}"))

## Helpers: servers, evaluation, gates

In [ ]:
SERVER = {"proc": None}

def start_server(cmd, name, timeout=1200):
    """Start an OpenAI-compatible server and wait until it answers. Logs go to runs/<name>-server.log."""
    stop_server()
    log = open(RUNS / f"{name}-server.log", "w")
    log.write(" ".join(map(str, cmd)) + "\n"); log.flush()
    SERVER["proc"] = subprocess.Popen(list(map(str, cmd)), stdout=log, stderr=subprocess.STDOUT)
    SERVER["name"] = name
    started = time.time()
    while time.time() - started < timeout:
        if SERVER["proc"].poll() is not None:
            raise RuntimeError(f"server exited; see runs/{name}-server.log:\n" + tail(name))
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=5)
            print(f"{name}: ready after {time.time() - started:.0f} s")
            return
        except OSError:
            time.sleep(5)
    raise TimeoutError(f"{name} did not start; see runs/{name}-server.log")

def stop_server():
    proc = SERVER.get("proc")
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(60)
        except subprocess.TimeoutExpired:
            proc.kill()
    SERVER["proc"] = None
    time.sleep(5)

def tail(name, n=40):
    return "\n".join((RUNS / f"{name}-server.log").read_text(errors="replace").splitlines()[-n:])

def vllm_cmd(model, *extra, lora=True):
    cmd = [VENV / "bin" / "vllm", "serve", model, "--port", PORT, "--dtype", "half", "--max-model-len", 4096,
           "--gpu-memory-utilization", 0.90, "--seed", 0, "--generation-config", "vllm"]
    if lora:
        cmd += ["--enable-lora", "--lora-modules", f"sftv4={ADAPTER_PEFT}", "--max-lora-rank", 16]
    return cmd + list(extra)

def evaluate(name, backend, model, adapter="none"):
    """Blind v3.1.0, serially, as in the SFT-V4 report; then rescored with urgency-rules-v5 (all 120, and
    the 92 without a close training neighbour). Resumable: rerun to continue after an interruption."""
    out = RUNS / f"acc-{name}.json"
    subprocess.run([sys.executable, "-m", "crisisgraph_ml.triage_eval_v3", "--backend", backend,
                    "--base-url", BASE_URL, "--model", model, "--adapter", str(adapter), "--dataset", DATASET,
                    "--prompt", PROMPT, "--schema", SCHEMA, "--max-tokens", "768", "--output", out], check=True)
    for suffix, extra in (("rules-v5", []), ("rules-v5-no-neighbour", ["--exclude-from", LEAK_AUDIT])):
        subprocess.run([sys.executable, "-c", "from crisisgraph_ml.triage_eval_v3 import main_rescore; main_rescore()",
                        out, "--dataset", DATASET, "--rules", "urgency-rules-v5",
                        "--output", RUNS / f"acc-{name}-{suffix}.json", *extra], check=True)
    return gate_check(name)

def gate_check(name):
    from crisisgraph_ml.triage_eval_v3 import headline
    head = headline(json.loads((RUNS / f"acc-{name}-rules-v5.json").read_text()))
    value = lambda k: len(head[k]) if isinstance(head[k], list) else head[k]   # some measures are lists of case ids
    passed = {k: (value(k) >= v if op == ">=" else value(k) <= v) for k, (op, v) in GATES.items()}
    head["gates_passed"] = passed
    head["all_gates_passed"] = all(passed.values())
    (RUNS / f"acc-{name}-headline.json").write_text(json.dumps(head, indent=2))
    print(json.dumps(head, indent=2))
    return head

def bench(name, model, backend="vllm", concurrency="1,2,4,8,16,32", per_level=64, reference=None,
          limit=0, notes=""):
    """Speed only (streaming). Answers are fingerprinted; with reference=..., changed answers are listed."""
    cmd = [sys.executable, "-m", "crisisgraph_ml.serving_bench", "--base-url", BASE_URL, "--backend", backend,
           "--model", model, "--dataset", DATASET, "--prompt", PROMPT, "--schema", SCHEMA,
           "--concurrency", concurrency, "--requests-per-level", str(per_level), "--gpu-sample",
           "--label", name, "--server-notes", notes, "--raw", RUNS / f"bench-{name}.jsonl",
           "--output", RUNS / f"bench-{name}.json"]
    if limit:
        cmd += ["--limit", str(limit)]
    if reference:
        cmd += ["--reference", RUNS / f"bench-{reference}.json"]
    subprocess.run(list(map(str, cmd)), check=True)
    return json.loads((RUNS / f"bench-{name}.json").read_text())

## 2. B-guidance: accuracy (blind v3.1.0, serial)
The B server with one change: `--structured-outputs-config '{"backend": "guidance"}'`. Then speed at
concurrency 1, 8 and 32 on the same server, so the speed and the accuracy come from one run.
Writes `partial.zip` afterwards.

In [ ]:
GUIDANCE = ["--structured-outputs-config", json.dumps({"backend": "guidance"})]
if RUN["B_guidance"]:
    start_server(vllm_cmd(BASE_DIR, *GUIDANCE), "B-guidance")
    head_Bg = evaluate("B-guidance", "vllm", "sftv4")
    bench_Bg = bench("B-guidance", "sftv4", concurrency="1,8,32",
                     notes=" ".join(map(str, vllm_cmd(BASE_DIR, *GUIDANCE))))
    stop_server()
    shutil.make_archive(str(WORK / "partial"), "zip", RUNS)
else:
    print('skipped: RUN["B_guidance"] is False')

## 3. Variant A: llama.cpp CUDA reference (the locally validated files)
Built from tag `b11429`, the build of the local SFT-V4 evaluation, with CUDA for the T4 (about 10-15 minutes).
One slot, as locally; speed at concurrency 1 only (it is the quality reference, not a throughput competitor).

In [ ]:
if RUN["A"]:
    try:
        LLAMA = SCRATCH / "llama.cpp"
        SERVER_BIN = LLAMA / "build" / "bin" / "llama-server"
        if not SERVER_BIN.exists():
            sh(f"git clone --depth 1 --branch {LLAMA_CPP_TAG} https://github.com/ggml-org/llama.cpp {LLAMA}")
            commit = sh(f"git -C {LLAMA} rev-parse HEAD")
            assert commit.startswith(LLAMA_CPP_COMMIT_PREFIX), commit
            sh(f"cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF")
            sh(f"cmake --build {LLAMA}/build --target llama-server -j {os.cpu_count()}")
        from huggingface_hub import hf_hub_download
        GGUF = Path(hf_hub_download(GGUF_REPO, GGUF_FILE, revision=GGUF_REV, local_dir=SCRATCH / "gguf"))
        assert hashlib.sha256(GGUF.read_bytes()).hexdigest() == GGUF_SHA256
        env["llama_cpp"] = {"tag": LLAMA_CPP_TAG, "commit": sh(f"git -C {LLAMA} rev-parse HEAD")}
        start_server([SERVER_BIN, "--model", GGUF, "--lora", ADAPTER_GGUF, "--alias", "local", "--host", "127.0.0.1",
                      "--port", PORT, "--ctx-size", 4096, "--parallel", 1, "-ngl", 99], "A")
        head_A = evaluate("A", "llama-cpp", "local", adapter="adapter-f16.gguf")
        bench_A = bench("A", "local", backend="llama-cpp", concurrency="1", per_level=32)
        stop_server()
    except Exception as error:   # record and continue to the summary
        env.setdefault("step_errors", {})['A'] = str(error)[:2000]
        print("STEP FAILED, continuing:", str(error)[:500])
        stop_server()
else:
    print("skipped: RUN[\"A\"] is False")

## 4. Summary and download

In [ ]:
summary = {"environment": env, "variants": {}}
for name in ("B-guidance", "A"):
    entry = {}
    h = RUNS / f"acc-{name}-headline.json"
    if h.exists():
        entry["accuracy"] = json.loads(h.read_text())
    b = RUNS / f"bench-{name}.json"
    if b.exists():
        entry["speed"] = {lvl["concurrency"]: {k: lvl[k] for k in ("requests_per_s", "output_tokens_per_s", "failed")}
                          | {"e2e_p50": lvl["e2e_s"]["p50"], "e2e_p95": lvl["e2e_s"]["p95"],
                             "ttft_p50": lvl["ttft_s"]["p50"], "gpu_mib": lvl.get("gpu_memory_peak_mib")}
                          for lvl in json.loads(b.read_text())["levels"]}
    if entry:
        summary["variants"][name] = entry
for n, v in summary["variants"].items():
    acc = v.get("accuracy", {})
    print(n, "gates:", acc.get("all_gates_passed"), json.dumps(acc.get("gates_passed")),
          "| invented", acc.get("invented_fact_cases"), "| V2 exact", acc.get("v2_exact_match"))
    for c, s in v.get("speed", {}).items():
        print(f"   c{c}: {60 * s['requests_per_s']:.1f}/min, e2e p50 {s['e2e_p50']:.1f} s")
print("step errors:", env.get("step_errors"))
(RUNS / "environment.json").write_text(json.dumps(env, indent=2))
(RUNS / "summary.json").write_text(json.dumps(summary, indent=2))
stop_server()
shutil.make_archive(str(WORK / "ml-s1-session2-results"), "zip", RUNS)
print("download /kaggle/working/ml-s1-session2-results.zip")